In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_design = pd.read_csv("input_files/210723_ECpilotCRISPRi.design.txt", sep="\t")
df_counts = pd.read_csv("input_files/GuideCounts.flat.tsv", sep="\t")

In [ ]:
# use this to get oligo ID and corresponding target gene
df_design

In [ ]:
# use this to get counts of each oligo ID in different samples
df_counts['sample_num'] = df_counts['SampleID'].str.split("-").str[1]
df_counts

In [ ]:
mapping = {
    1: "Dox0-D1",
    4: "Dox0-D1",
    7: "Dox0-D1",
    8: "Dox0-D1",
    2: "Dox0-D4_high",
    5: "Dox0-D4_high",
    9: "Dox0-D4_high",
    11: "Dox0-D4_high",
    3: "Dox0-D4_low",
    6: "Dox0-D4_low",
    10: "Dox0-D4_low",
    12: "Dox0-D4_low",
    13: "Dox24-D1",
    16: "Dox24-D1",
    19: "Dox24-D1",
    20: "Dox24-D1",
    14: "Dox24-D4_high",
    17: "Dox24-D4_high",
    21: "Dox24-D4_high",
    23: "Dox24-D4_high",
    15: "Dox24-D4_low",
    18: "Dox24-D4_low",
    22: "Dox24-D4_low",
    24: "Dox24-D4_low"
}

df_counts["bins"] = df_counts["sample_num"].astype(int).map(mapping)
df_counts

In [ ]:
# sum the counts of oligos in each bin
df_final = df_counts.groupby(["bins", "OligoID"])["count"].sum().unstack(0)

# add columns for D4 total
df_final["Dox0-D4"] = df_final["Dox0-D4_high"] + df_final["Dox0-D4_low"]
df_final["Dox24-D4"] = df_final["Dox24-D4_high"] + df_final["Dox24-D4_low"]
df_final

In [ ]:
# get OligoID's target gene name
df_merged = (
    pd.merge(
        left=df_final,
        right=df_design[["OligoID", "guideSet"]],
        left_on="OligoID",
        right_on="OligoID", 
        how="left")
    .sort_values(by=["OligoID"])
)

# rearrange columns
df_merged = df_merged[["OligoID", "guideSet", "Dox0-D1", "Dox0-D4", "Dox0-D4_high", "Dox0-D4_low", "Dox24-D1", "Dox24-D4", "Dox24-D4_high", "Dox24-D4_low"]]
df_merged = df_merged.rename(columns = {'guideSet': 'gene'})

# sort by OligoID
df_merged["num"] = df_merged["OligoID"].str.split("_").str[2]
df_merged.num = df_merged.num.astype(float)
df_merged = df_merged.sort_values("num")
df_merged = df_merged.drop(["num"], axis=1)

In [ ]:
df_merged

In [ ]:
df_merged.to_csv("output_files/sgrna_counts.txt", sep="\t", header=True, index=False)